# Hybrid First-Sentence Summarization — IndoBERT + SeaLLM

Pipeline:
**First Sentence Premise → IndoBERT Textual Entailment → Evidence Selection → SeaLLM Evidence-Grounded Paraphrase → Entailment/Fact Verification → Final Summary**

Notebook ini dirancang untuk eksperimen ringkasan berita Bahasa Indonesia dengan prinsip:

- **Evidence-grounded**: SeaLLM hanya boleh menggunakan informasi dari evidence yang diberikan.
- **Anti-hallucination**: tidak boleh menambah fakta, angka, tanggal, lokasi, tokoh, institusi, sebab-akibat, atau kutipan yang tidak didukung evidence.
- **Fact preservation**: nama, angka, tanggal, organisasi, lokasi, dan atribusi dijaga.
- **Completeness-oriented**: fakta utama dari evidence dipertahankan sebanyak mungkin tanpa membuat ringkasan menjadi panjang.
- **Coherence-oriented**: kalimat dirangkai secara alami, kronologis/logis, tanpa lompatan referen.
- **Deterministic generation**: sampling dimatikan agar hasil eksperimen lebih stabil dan reproducible.

> Catatan penelitian: prompt engineering membantu mengarahkan model, tetapi tidak dapat menjamin factuality sempurna. Karena itu notebook ini menambahkan verifikasi pascagenerasi berbasis IndoBERT + pemeriksaan fakta eksplisit.

In [1]:
import os
import re
import json
import math
import random
import numpy as np
import pandas as pd
import torch
import nltk
import jsonlines

from tqdm.auto import tqdm
from tabulate import tabulate
from nltk.tokenize import sent_tokenize, word_tokenize
from difflib import SequenceMatcher

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BertForSequenceClassification
)

print("Python / Torch environment loaded.")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

d:\conda_envs\nlp_project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python / Torch environment loaded.
PyTorch: 2.8.0+cu128
CUDA available: True


In [2]:
# ============================================================
# 1. KONFIGURASI EKSPERIMEN
# ============================================================

CHECKPOINT_ENT_PATH = r"D:\textualentailments\modelte\BERT-20250925T085910Z-1-001\BERT\results_indobert\checkpoint-343355"

# SeaLLM multilingual model.
# v2.5 is supported through Transformers chat_template.
# Bisa diganti menjadi "SeaLLMs/SeaLLM-7B-v2" bila eksperimen ingin
# mempertahankan versi v2 secara eksplisit.
SEALLM_MODEL_NAME = "SeaLLMs/SeaLLM-7B-v2.5"

DATA_FILE = r"D:\textualentailments\liputan6_dataset_cleaned_dataset.jsonl"
OUTPUT_FILE = r"D:\textualentailments\1_final_summary_first_sentences_liputan6_seallm.jsonl"

NUM_DOCS = 100
RATIOS = [0.25, 0.50, 0.75, 1.00]

# Ubah menjadi angka kecil untuk debug sebelum menjalankan 1000 dokumen.
TEST_DOCS = 10
RUN_FULL_EXPERIMENT = True

# Verifikasi summary menggunakan IndoBERT.
SUMMARY_ENTAILMENT_THRESHOLD = 0.55

# Batas evidence yang masuk ke SeaLLM.
MAX_INPUT_TOKENS = 1500

# Batas output SeaLLM.
MAX_NEW_TOKENS = 180

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
    else:
        DTYPE = torch.float16
else:
    DTYPE = torch.float32

print(f"Device : {device}")
print(f"Dtype  : {DTYPE}")

Device : cuda
Dtype  : torch.bfloat16


In [3]:
# ============================================================
# 2. NLTK RESOURCE
# ============================================================

for resource in ["punkt", "punkt_tab"]:
    try:
        nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        try:
            nltk.download(resource, quiet=True)
        except Exception:
            pass

print("✅ NLTK ready.")

✅ NLTK ready.


## 3. Prompt Engineering: prinsip inti

Prompt SeaLLM menggunakan beberapa lapisan instruksi:

1. **Role** — model diposisikan sebagai editor ringkasan berita Bahasa Indonesia.
2. **Closed-book constraint** — model dilarang menggunakan pengetahuan luar evidence.
3. **Evidence hierarchy** — evidence menjadi satu-satunya sumber kebenaran.
4. **Fact-preservation rules** — nama, angka, tanggal, lokasi, organisasi, dan atribusi harus dipertahankan.
5. **No inference rule** — hubungan sebab-akibat, identitas, status, dan kronologi tidak boleh disimpulkan jika tidak tertulis.
6. **Completeness rule** — fakta utama diprioritaskan, bukan sekadar menyalin kalimat yang paling mirip.
7. **Coherence rule** — hasil harus menjadi ringkasan berita yang utuh, bukan kumpulan fragmen.
8. **Output contract** — hanya ringkasan, tanpa analisis, disclaimer, label, atau komentar.

Prompt juga diberi **critical facts** yang diekstraksi dari evidence. Tujuannya membantu model menjaga angka, tanggal, nama, singkatan, dan entitas penting tanpa meminta model mengeluarkan chain-of-thought.

In [4]:
# ============================================================
# 3. TEXT CLEANING & BASIC UTILITIES
# ============================================================

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,!?])", r"\1", text)
    return text.strip()


def remove_repetition(text: str) -> str:
    words = text.split()
    result = []

    for i, w in enumerate(words):
        if i > 0 and w.lower() == words[i - 1].lower():
            continue
        result.append(w)

    return " ".join(result)


def fix_sentences(text: str) -> str:
    text = clean_text(text)
    sentences = sent_tokenize(text)
    fixed = []

    for s in sentences:
        s = s.strip()
        if not s:
            continue

        s = s[0].upper() + s[1:] if len(s) > 1 else s.upper()

        if not s.endswith((".", "!", "?")):
            s += "."

        fixed.append(s)

    return " ".join(fixed)


def clean_model_output(text: str) -> str:
    if not text:
        return ""

    # Remove accidental wrapper labels commonly produced by LLMs.
    text = re.sub(
        r"(?i)^\s*(ringkasan|summary|parafrase|paraphrase)\s*[:\-]\s*",
        "",
        text
    )

    # Remove markdown fences / bullets.
    text = text.replace("```", "")
    text = re.sub(r"(?m)^\s*[-*•]\s*", "", text)

    # Remove obvious instruction leakage.
    text = re.sub(r"(?i)\b(berikut adalah|tentu,|baik, berikut)\s*:?\s*", "", text)

    # Safe punctuation normalization.
    text = re.sub(r"\s+([,.:;!?%])", r"\1", text)
    text = re.sub(r"([,!?])\1+", r"\1", text)
    text = re.sub(r"\.{2,}", ".", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text


def smart_sentence_split(text: str) -> str:
    sentences = sent_tokenize(clean_text(text))
    final = []

    for s in sentences:
        s = s.strip().rstrip(",;:").strip()

        if not s:
            continue

        s = s[0].upper() + s[1:] if len(s) > 1 else s.upper()

        if not s.endswith((".", "!", "?")):
            s += "."

        final.append(s)

    return remove_repetition(" ".join(final))

In [5]:
# ============================================================
# 4. EXPLICIT FACT EXTRACTION
#    Dipakai sebagai "guardrail" untuk prompt & evaluasi.
# ============================================================

MONTHS = (
    "Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|"
    "Oktober|November|Desember"
)

def extract_critical_facts(text: str) -> dict:
    text = clean_text(text)

    numbers = re.findall(
        r"(?<!\w)(?:\d{1,3}(?:[.,]\d{1,3})*|\d+)(?:\s?%|\s?(?:juta|miliar|triliun|ribu|km|kg|orang|tahun|WIB|WITA|WIT))?",
        text,
        flags=re.IGNORECASE
    )

    dates = re.findall(
        rf"\b\d{{1,2}}\s+(?:{MONTHS})\s+\d{{4}}\b",
        text,
        flags=re.IGNORECASE
    )

    # Time values and date-like numeric forms.
    times = re.findall(
        r"\b\d{1,2}[.:]\d{2}\s?(?:WIB|WITA|WIT)?\b",
        text,
        flags=re.IGNORECASE
    )

    # Acronyms such as BMKG, DKI, SPPG, SLHS, PPPA.
    acronyms = re.findall(r"\b[A-Z]{2,8}\b", text)

    # Capitalized multiword entities. Heuristic only.
    proper_nouns = re.findall(
        r"\b(?:[A-Z][a-zA-ZÀ-ÿ'-]+)(?:\s+[A-Z][a-zA-ZÀ-ÿ'-]+){1,5}\b",
        text
    )

    # Deduplicate while preserving order.
    def uniq(items):
        seen = set()
        out = []
        for x in items:
            x = x.strip()
            key = x.lower()
            if x and key not in seen:
                seen.add(key)
                out.append(x)
        return out

    return {
        "numbers": uniq(numbers),
        "dates": uniq(dates),
        "times": uniq(times),
        "acronyms": uniq(acronyms),
        "proper_nouns": uniq(proper_nouns),
    }


def format_critical_facts(facts: dict) -> str:
    rows = []

    for label, key in [
        ("ANGKA", "numbers"),
        ("TANGGAL", "dates"),
        ("WAKTU", "times"),
        ("SINGKATAN", "acronyms"),
        ("ENTITAS/NAMA", "proper_nouns"),
    ]:
        values = facts.get(key, [])
        if values:
            rows.append(f"- {label}: {', '.join(values[:30])}")

    return "\n".join(rows) if rows else "- Tidak ada fakta eksplisit tambahan."

In [6]:
# ============================================================
# 5. LOAD INDOBERT ENTAILMENT MODEL
# ============================================================

print("🔄 Loading IndoBERT Entailment Model...")

ent_tok = AutoTokenizer.from_pretrained(
    CHECKPOINT_ENT_PATH,
    trust_remote_code=True
)

ent_mdl = BertForSequenceClassification.from_pretrained(
    CHECKPOINT_ENT_PATH,
    trust_remote_code=True
)

ent_mdl.to(device)
ent_mdl.eval()

print("✅ IndoBERT loaded.")

🔄 Loading IndoBERT Entailment Model...
✅ IndoBERT loaded.


In [7]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
# ============================================================
# 6. LOAD SEALLM — 4-BIT FOR RTX 3060
# ============================================================

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print(f"🔄 Loading {SEALLM_MODEL_NAME} ...")

# ------------------------------------------------------------
# 1. Tokenizer
# ------------------------------------------------------------
seallm_tok = AutoTokenizer.from_pretrained(
    SEALLM_MODEL_NAME
)

# ------------------------------------------------------------
# 2. 4-bit Quantization
# ------------------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# ------------------------------------------------------------
# 3. Model
# ------------------------------------------------------------
seallm_mdl = AutoModelForCausalLM.from_pretrained(
    SEALLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

# ------------------------------------------------------------
# 4. Pad token
# ------------------------------------------------------------
if seallm_tok.pad_token_id is None:
    seallm_tok.pad_token = seallm_tok.eos_token

print("✅ SeaLLM loaded!")
print(f"📌 Model: {SEALLM_MODEL_NAME}")
print(f"📌 Device: {seallm_mdl.device}")
print("📌 Quantization: 4-bit NF4")

🔄 Loading SeaLLMs/SeaLLM-7B-v2.5 ...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# 7. INDOBERT ENTAILMENT FUNCTION
# ============================================================

def check_entailment(premise: str, hypothesis: str):
    inputs = ent_tok(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = ent_mdl(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=-1)

    entailment_score = probs[0][0].item()
    predicted_label = torch.argmax(logits, dim=-1).item()

    # Dataset/model notebook saat ini mendefinisikan label 0 = entailment.
    is_entailment = (predicted_label == 0)

    return is_entailment, entailment_score


def best_source_support(summary_sentence: str, evidence_sentences: list):
    best_score = 0.0
    best_source = ""

    for src in evidence_sentences:
        try:
            _, score = check_entailment(src, summary_sentence)
        except Exception:
            continue

        if score > best_score:
            best_score = score
            best_source = src

    return best_score, best_source

## 8. Prompt SeaLLM — berita yang evidence-grounded

Prompt berikut sengaja **tidak meminta model mengarang atau "memperbaiki" fakta**. Model diberi mandat sebagai *evidence-grounded news editor*.

Prioritas isi:
**faktualitas > keterikatan pada evidence > kelengkapan fakta utama > koherensi > keluwesan paraphrase**.

Artinya, ketika parafrase yang lebih indah berpotensi mengubah fakta, model harus memilih formulasi yang lebih konservatif.

In [ ]:
# ============================================================
# 8. ADVANCED NEWS PROMPT
# ============================================================

SYSTEM_PROMPT = r'''
Anda adalah editor berita Bahasa Indonesia yang bertugas melakukan
PARAFRASE ABSTRAKTIF TERKENDALI.

TUGAS UTAMA
Tulis satu ringkasan berita yang ringkas, faktual, lengkap terhadap
fakta utama, koheren, dan seluruhnya BERDASARKAN EVIDENCE yang diberikan.

SUMBER KEBENARAN
1. EVIDENCE adalah satu-satunya sumber kebenaran.
2. Jangan menggunakan pengetahuan umum, ingatan model, asumsi,
   konteks berita lain, atau informasi dunia di luar EVIDENCE.
3. Setiap klaim dalam ringkasan harus dapat ditelusuri kembali ke
   satu atau lebih kalimat EVIDENCE.
4. Bila suatu informasi tidak terdapat secara eksplisit dalam EVIDENCE,
   JANGAN memasukkannya.
5. Bila dua informasi tampak bertentangan, jangan memilih salah satunya
   berdasarkan pengetahuan luar. Pertahankan hanya formulasi yang
   benar-benar didukung EVIDENCE.

ATURAN ANTI-HALUSINASI / ANTI-INFERENSI
- DILARANG menambahkan tokoh, organisasi, tempat, tanggal, angka,
  jumlah korban, jabatan, status, penyebab, akibat, motif, atau detail lain.
- DILARANG mengubah angka, tanggal, waktu, lokasi, nama diri,
  nama institusi, singkatan, atau istilah teknis.
- DILARANG membuat hubungan sebab-akibat baru.
- DILARANG mengubah dugaan menjadi fakta.
- DILARANG mengubah pernyataan menjadi kesimpulan yang lebih kuat.
- DILARANG menciptakan kutipan langsung.
- Bila EVIDENCE memuat atribusi seperti "menurut", "kata",
  "menyebutkan", atau "meminta", pertahankan sumber/atribusi tersebut
  bila relevan terhadap klaim.
- Jangan menggabungkan dua subjek berbeda menjadi satu subjek.
- Jangan menganggap kata ganti (dia, mereka, ini, tersebut) merujuk
  pada entitas tertentu bila EVIDENCE tidak cukup jelas.

ATURAN FAKTUALITAS
- Nama dan entitas penting harus dipertahankan.
- Angka dan unit harus dipertahankan.
- Tanggal dan waktu harus dipertahankan bila menjadi bagian fakta utama.
- Lokasi penting harus dipertahankan.
- Jabatan atau organisasi harus dipertahankan bila menjadi bagian klaim.
- Gunakan istilah yang setara hanya jika maknanya tetap sama.
- Jangan membuat sinonim yang mengubah makna hukum, statistik,
  administratif, politik, atau medis.

ATURAN KELENGKAPAN
Prioritaskan informasi dengan urutan:
1. peristiwa/topik utama,
2. aktor utama,
3. tindakan/pernyataan utama,
4. waktu dan lokasi penting,
5. angka/statistik penting,
6. dampak/konsekuensi yang SECARA EKSPLISIT disebut,
7. konteks tambahan yang diperlukan agar ringkasan tetap utuh.

Jangan mengejar kelengkapan dengan cara menyalin seluruh EVIDENCE.
Pilih fakta utama yang paling informatif.

ATURAN KOHERENSI
- Hasil harus terasa sebagai satu ringkasan berita yang utuh.
- Gunakan urutan informasi yang logis dan alami.
- Hindari lompatan topik mendadak.
- Hindari pengulangan fakta.
- Pastikan referen tokoh/organisasi jelas.
- Gabungkan kalimat hanya bila hubungan logisnya benar-benar didukung.
- Jangan memakai konjungsi kausal seperti "sehingga", "karena itu",
  "akibatnya", atau "oleh sebab itu" kecuali hubungan tersebut memang
  dinyatakan atau didukung jelas oleh EVIDENCE.

ATURAN GAYA
- Bahasa Indonesia baku, natural, dan bergaya berita.
- Jangan menggunakan bahasa promosi atau opini.
- Jangan menambahkan pembuka seperti "Ringkasnya:".
- Jangan menambahkan komentar tentang tugas.
- Jangan menyebut kata "evidence", "prompt", "model", atau "instruksi"
  dalam hasil akhir.
- Jangan menggunakan bullet list.
- Output hanya teks ringkasan.
'''


def build_user_prompt(evidence: str, critical_facts: dict, ratio: float) -> str:
    words = len(word_tokenize(evidence)) if evidence else 0

    if ratio <= 0.25:
        target = "1–2 kalimat; fokus pada inti peristiwa dan fakta paling penting."
        word_budget = (35, 70)
    elif ratio <= 0.50:
        target = "1–3 kalimat; pertahankan inti peristiwa dan fakta pendukung penting."
        word_budget = (40, 85)
    elif ratio <= 0.75:
        target = "2–4 kalimat; pertahankan lebih banyak detail penting dari evidence."
        word_budget = (45, 100)
    else:
        target = "2–5 kalimat; prioritaskan kelengkapan fakta utama tanpa menyalin seluruh evidence."
        word_budget = (50, 120)

    fact_block = format_critical_facts(critical_facts)

    return f'''
HASIL YANG DIMINTA
Buat ringkasan berita Bahasa Indonesia berdasarkan EVIDENCE di bawah.

TARGET PANJANG
{target}
Target sekitar {word_budget[0]}–{word_budget[1]} kata bila memungkinkan.
Jangan memaksakan jumlah kata jika hal itu menyebabkan fakta penting hilang.

KONTEKS PANJANG EVIDENCE
Evidence berisi sekitar {words} kata.

FAKTA KRITIS YANG HARUS DIJAGA
{fact_block}

CHECKLIST SEBELUM MENULIS
- Siapa/apa peristiwa utama?
- Tindakan atau pernyataan utamanya apa?
- Di mana dan kapan bila disebutkan?
- Angka/nama/organisasi penting apa yang wajib dipertahankan?
- Apa dampak/konteks penting yang benar-benar ada di evidence?
- Apakah setiap klaim dapat ditelusuri ke evidence?
- Apakah ada klaim baru yang hanya berasal dari pengetahuan model?

ATURAN FINAL
Jangan menjelaskan proses berpikir.
Jangan menulis analisis.
Jangan menulis daftar fakta.
Jangan membuat informasi baru.
Tampilkan HANYA ringkasan berita final.

EVIDENCE:
{evidence}
'''.strip()


def generate_with_seallm(evidence: str, ratio: float) -> str:
    critical_facts = extract_critical_facts(evidence)
    user_prompt = build_user_prompt(evidence, critical_facts, ratio)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {"role": "user", "content": user_prompt}
    ]

    inputs = seallm_tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    target_device = next(seallm_mdl.parameters()).device
    inputs = {k: v.to(target_device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = seallm_mdl.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=3,
            no_repeat_ngram_size=4,
            repetition_penalty=1.0,
            length_penalty=1.05,
            early_stopping=True,
            pad_token_id=seallm_tok.pad_token_id,
            eos_token_id=seallm_tok.eos_token_id
        )

    input_len = inputs["input_ids"].shape[-1]
    generated_ids = output_ids[0][input_len:]

    raw = seallm_tok.decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return smart_sentence_split(clean_model_output(raw))


def contains_fact(value: str, text: str) -> bool:
    if not value:
        return True
    return value.lower() in text.lower()


def critical_fact_coverage(summary: str, evidence: str) -> float:
    facts = extract_critical_facts(evidence)
    required = []

    required.extend(facts["numbers"])
    required.extend(facts["dates"])
    required.extend(facts["times"])

    # Acronyms and proper nouns are only used as soft checks because
    # not every named entity is necessarily a core fact for the final summary.
    required.extend(facts["acronyms"][:10])

    required = [x for x in required if len(x.strip()) > 1]

    if not required:
        return 1.0

    hit = sum(1 for x in required if contains_fact(x, summary))
    return hit / len(required)

In [ ]:
# ============================================================
# 9. SEA LLM RETRY / QUALITY GUARD
# ============================================================

def lexical_relevance_score(summary: str, evidence: str) -> float:
    summary_words = set(re.findall(r"\b[\w'-]+\b", summary.lower()))
    evidence_words = set(re.findall(r"\b[\w'-]+\b", evidence.lower()))

    # Remove generic function words for a more useful overlap diagnostic.
    stop = {
        "dan","yang","di","ke","dari","untuk","dengan","pada","dalam","ini",
        "itu","sebagai","adalah","akan","telah","dapat","juga","karena",
        "oleh","atau","sehingga","bahwa","agar","para","sejumlah"
    }

    summary_words -= stop
    evidence_words -= stop

    if not summary_words:
        return 0.0

    return len(summary_words & evidence_words) / len(summary_words)


def summarize_with_retry(evidence: str, ratio: float, max_attempts: int = 2) -> dict:
    last_summary = ""

    for attempt in range(max_attempts):
        summary = generate_with_seallm(evidence, ratio)
        last_summary = summary

        coverage = critical_fact_coverage(summary, evidence)
        relevance = lexical_relevance_score(summary, evidence)

        sentences = sent_tokenize(summary)
        support_scores = []

        evidence_sents = sent_tokenize(evidence)

        for s in sentences:
            score, _ = best_source_support(s, evidence_sents)
            support_scores.append(score)

        avg_support = float(np.mean(support_scores)) if support_scores else 0.0
        weak_sentences = sum(
            1 for x in support_scores if x < SUMMARY_ENTAILMENT_THRESHOLD
        )

        # Strict acceptance:
        # - at least moderate evidence support,
        # - reasonable critical-fact preservation,
        # - non-trivial relevance,
        # - no unsupported sentence dominating the output.
        accepted = (
            bool(summary.strip())
            and coverage >= 0.50
            and relevance >= 0.20
            and avg_support >= SUMMARY_ENTAILMENT_THRESHOLD
            and weak_sentences == 0
        )

        if accepted:
            return {
                "summary": summary,
                "coverage": coverage,
                "relevance": relevance,
                "avg_entailment_support": avg_support,
                "weak_sentence_count": weak_sentences,
                "attempt": attempt + 1,
                "accepted": True
            }

    return {
        "summary": last_summary,
        "coverage": critical_fact_coverage(last_summary, evidence),
        "relevance": lexical_relevance_score(last_summary, evidence),
        "avg_entailment_support": 0.0,
        "weak_sentence_count": 0,
        "attempt": max_attempts,
        "accepted": False
    }

In [ ]:
# ============================================================
# 10. ENTailMENT EXTRACTION
# ============================================================

def extract_entailment(doc_group: pd.DataFrame):
    premise_row = doc_group[
        (doc_group["paragraph_hypothesis"] == 1) &
        (doc_group["sentence_hypothesis"] == 1)
    ]

    if premise_row.empty:
        return None

    premise = premise_row.iloc[0]["hypothesis"]

    if not isinstance(premise, str) or not premise.strip():
        return None

    full_content = (
        doc_group.iloc[0]["content"]
        if "content" in doc_group.columns else ""
    )

    full_content = full_content if isinstance(full_content, str) else ""

    pre_sentences = sent_tokenize(full_content) if full_content else []
    pre_sent_count = len(pre_sentences)
    pre_word_count = len(word_tokenize(full_content)) if full_content else 0
    pre_char_count = len(full_content)

    candidates = (
        doc_group[["hypothesis"]]
        .dropna()
        .reset_index(drop=True)
    )

    total_sentences = (
        pre_sent_count if pre_sent_count > 0 else len(candidates)
    )

    if len(candidates) <= 1:
        return None

    entailed_texts = []
    entailment_details = []

    for idx, row in candidates.iloc[1:].iterrows():
        hyp = row["hypothesis"]

        if not isinstance(hyp, str) or not hyp.strip():
            continue

        is_ent, score = check_entailment(premise, hyp)

        if is_ent:
            entailed_texts.append((score, hyp))
            entailment_details.append({
                "position": idx + 1,
                "text": hyp,
                "score": score
            })

    if not entailed_texts:
        return None

    entailment_pct = (len(entailed_texts) / total_sentences) * 100
    avg_confidence = float(
        np.mean([x[0] for x in entailed_texts])
    )

    # Keep confidence-based ordering as in the original notebook.
    sorted_entailed = sorted(
        entailed_texts,
        key=lambda x: x[0],
        reverse=True
    )

    return {
        "doc_id": doc_group.iloc[0]["doc_id"],
        "title": doc_group.iloc[0]["title"],
        "content": full_content,
        "premise": premise,
        "sorted_entailed": sorted_entailed,
        "entailment_count": len(entailed_texts),
        "total_sentences": total_sentences,
        "entailment_percentage": entailment_pct,
        "avg_confidence": avg_confidence,
        "entailment_details": entailment_details,
        "pre_sent_count": pre_sent_count,
        "pre_word_count": pre_word_count,
        "pre_char_count": pre_char_count,
    }

In [ ]:
# ============================================================
# 11. SUMMARIZATION PER RATIO
# ============================================================

def prepare_evidence(cache: dict, compression_ratio: float):
    sorted_entailed = cache["sorted_entailed"]

    k = max(
        1,
        math.floor(len(sorted_entailed) * compression_ratio)
    )

    selected = [text for _, text in sorted_entailed[:k]]

    # First sentence remains mandatory because it is the selected premise.
    selected_texts = [cache["premise"]] + selected

    evidence = " ".join(selected_texts)

    # Token-safe truncation for SeaLLM.
    tokenized = seallm_tok(
        evidence,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
        return_tensors=None
    )

    input_ids = tokenized["input_ids"]
    if len(input_ids) > MAX_INPUT_TOKENS:
        input_ids = input_ids[:MAX_INPUT_TOKENS]
        evidence = seallm_tok.decode(
            input_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

    return k, selected, evidence


def summarize_with_ratio(cache: dict, compression_ratio: float) -> dict:
    k, selected_texts, evidence = prepare_evidence(
        cache,
        compression_ratio
    )

    result = summarize_with_retry(
        evidence,
        compression_ratio,
        max_attempts=2
    )

    summary = result["summary"]

    return {
        **cache,
        "abstractive_summary": summary,
        "compression_ratio": compression_ratio,
        "entailment_selected": k,
        "selected_evidence": selected_texts,
        "evidence": evidence,
        "post_sent_count": len(sent_tokenize(summary)),
        "post_word_count": len(word_tokenize(summary)),
        "post_char_count": len(summary),
        "critical_fact_coverage": result["coverage"],
        "lexical_relevance": result["relevance"],
        "avg_entailment_support": result["avg_entailment_support"],
        "weak_sentence_count": result["weak_sentence_count"],
        "generation_attempt": result["attempt"],
        "quality_accepted": result["accepted"],
    }


def process_document_group(doc_group: pd.DataFrame, compression_ratio: float = 1.0):
    cache = extract_entailment(doc_group)

    if cache is None:
        return None

    return summarize_with_ratio(
        cache,
        compression_ratio
    )

In [ ]:
# ============================================================
# 12. LOAD DATA
# ============================================================

print(f"📂 Loading: {DATA_FILE}")

data_list = []

with jsonlines.open(DATA_FILE, "r") as reader:
    for obj in reader:
        data_list.append(obj)

df = pd.DataFrame(data_list)

unique_doc_ids = df["doc_id"].unique()[:NUM_DOCS]
df = df[df["doc_id"].isin(unique_doc_ids)].copy()

print(f"✅ Total rows       : {len(df):,}")
print(f"✅ Unique documents : {len(unique_doc_ids):,}")

In [ ]:
# ============================================================
# 13. PREVIEW: PREMISE + HYPOTHESIS
# ============================================================

sample_doc_id = df["doc_id"].iloc[0]
doc_group = df[df["doc_id"] == sample_doc_id]

premise_row = doc_group[
    (doc_group["paragraph_hypothesis"] == 1) &
    (doc_group["sentence_hypothesis"] == 1)
]

if not premise_row.empty:
    premise = premise_row.iloc[0]["hypothesis"]

    hypotheses = (
        doc_group["hypothesis"]
        .dropna()
        .reset_index(drop=True)
    )

    preview_rows = [
        [sample_doc_id, premise, h]
        for h in hypotheses.iloc[1:11]
    ]

    print(tabulate(
        preview_rows,
        headers=["doc_id", "premise (first sentence)", "hypothesis"],
        tablefmt="fancy_grid",
        maxcolwidths=[16, 60, 70],
        showindex=False
    ))
else:
    print("Premise tidak ditemukan.")

In [ ]:
# ============================================================
# 14. DEBUG TEST SEBELUM FULL RUN
# ============================================================

debug_doc_ids = df["doc_id"].unique()[:TEST_DOCS]

debug_rows = []

for doc_id in tqdm(debug_doc_ids, desc="Debug documents"):
    group = df[df["doc_id"] == doc_id]
    cache = extract_entailment(group)

    if not cache:
        continue

    result = summarize_with_ratio(cache, 0.50)

    debug_rows.append({
        "doc_id": doc_id,
        "title": cache["title"],
        "summary": result["abstractive_summary"],
        "coverage": result["critical_fact_coverage"],
        "relevance": result["lexical_relevance"],
        "entailment_support": result["avg_entailment_support"],
        "accepted": result["quality_accepted"]
    })

debug_df = pd.DataFrame(debug_rows)

if not debug_df.empty:
    print(debug_df.to_string(index=False))
else:
    print("Tidak ada hasil debug.")

## 15. Full experiment

Urutan eksperimen tetap sama seperti notebook awal:

- IndoBERT dijalankan **1 kali per dokumen** untuk memperoleh cache entailment.
- Untuk setiap dokumen, evidence dipilih pada rasio **25%, 50%, 75%, 100%**.
- SeaLLM menjalankan parafrase abstraktif pada evidence tersebut.
- Hasil SeaLLM kemudian diperiksa kembali dengan IndoBERT dan *critical-fact checks*.
- Setiap rasio tetap disimpan agar dapat dibandingkan pada ROUGE dan human evaluation.

In [ ]:
# ============================================================
# 15. FULL EXPERIMENT
# ============================================================

if not RUN_FULL_EXPERIMENT:
    print("RUN_FULL_EXPERIMENT=False → full experiment dilewati.")
else:
    print("\n🔍 Tahap 1: Entailment cache (1x per dokumen)")

    entailment_cache_first_liputan = []

    for doc_id in tqdm(unique_doc_ids, desc="Entailment"):
        doc_group = df[df["doc_id"] == doc_id]
        cache = extract_entailment(doc_group)

        if cache:
            entailment_cache_first_liputan.append(cache)

    print(
        f"✅ Cache entailment selesai: "
        f"{len(entailment_cache_first_liputan)} dokumen"
    )

    results_by_ratio_first = {}

    for ratio in RATIOS:
        label = f"{int(ratio * 100)}%"

        print("\n" + "=" * 72)
        print(f"  🤖 SeaLLM Evidence-Grounded Paraphrase — Rasio {label}")
        print("=" * 72)

        results_ratio = []

        for cache in tqdm(
            entailment_cache_first_liputan,
            desc=f"SeaLLM Ratio {label}"
        ):
            result = summarize_with_ratio(
                cache,
                compression_ratio=ratio
            )

            if result:
                results_ratio.append(result)

        results_by_ratio_first[ratio] = results_ratio

        accepted = sum(
            1 for x in results_ratio
            if x["quality_accepted"]
        )

        print(
            f"✅ Berhasil: {len(results_ratio)} dokumen | "
            f"Quality guard accepted: {accepted}"
        )

    results = results_by_ratio_first[1.00]

    print(
        f"\n✅ Semua rasio selesai. "
        f"Default results (100%) = {len(results)} dokumen"
    )

In [ ]:
# ============================================================
# 16. SAVE OUTPUT JSONL
# ============================================================

if RUN_FULL_EXPERIMENT:
    TAG = "first_seallm"
    merged = {}

    for ratio, label in [
        (0.25, "25"),
        (0.50, "50"),
        (0.75, "75"),
        (1.00, "100")
    ]:
        for res in results_by_ratio_first[ratio]:
            doc_id = res["doc_id"]

            if doc_id not in merged:
                merged[doc_id] = {
                    "doc_id": res["doc_id"],
                    "title": res["title"],
                    "content": res["content"],
                    "used_premise": res["premise"],
                    "entailment_count": res["entailment_count"],
                    "total_sentences": res["total_sentences"],
                    "avg_confidence": res["avg_confidence"],
                }

            merged[doc_id][f"summary_{label}"] = res["abstractive_summary"]
            merged[doc_id][f"entailment_selected_{label}"] = res["entailment_selected"]
            merged[doc_id][f"critical_fact_coverage_{label}"] = res["critical_fact_coverage"]
            merged[doc_id][f"lexical_relevance_{label}"] = res["lexical_relevance"]
            merged[doc_id][f"avg_entailment_support_{label}"] = res["avg_entailment_support"]
            merged[doc_id][f"quality_accepted_{label}"] = res["quality_accepted"]
            merged[doc_id][f"generation_attempt_{label}"] = res["generation_attempt"]

    merged_list = list(merged.values())

    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for row in merged_list:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                ) + "\n"
            )

    print(f"💾 Saved: {OUTPUT_FILE}")
    print(f"📄 Documents: {len(merged_list)}")
else:
    print("Output tidak disimpan karena full experiment tidak dijalankan.")

In [ ]:
# ============================================================
# 17. STATISTICS
# ============================================================

def col_stats(series, label):
    series = pd.to_numeric(series, errors="coerce").dropna()

    if series.empty:
        return []

    return [
        [f"Rata-rata {label}", f"{series.mean():.4f}"],
        [f"Min {label}", f"{series.min():.4f}"],
        [f"Max {label}", f"{series.max():.4f}"],
        [f"Median {label}", f"{series.median():.4f}"],
        [f"Std Dev {label}", f"{series.std():.4f}"],
    ]


def print_table(title, rows):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)
    print(tabulate(
        rows,
        headers=["Metrik", "Nilai"],
        tablefmt="fancy_grid",
        colalign=("left", "right")
    ))


if RUN_FULL_EXPERIMENT:
    df_pre = pd.DataFrame(results_by_ratio_first[1.00])

    pre_rows = [
        ["Total Dokumen", len(df_pre)]
    ]

    pre_rows += col_stats(
        df_pre["pre_sent_count"],
        "Kalimat per Artikel"
    )

    pre_rows += col_stats(
        df_pre["pre_word_count"],
        "Kata per Artikel"
    )

    pre_rows += col_stats(
        df_pre["pre_char_count"],
        "Karakter per Artikel"
    )

    print_table(
        "STATISTIK SEBELUM EKSTRAKSI",
        pre_rows
    )

    ent_rows = [
        ["Total Dokumen Berhasil", len(df_pre)]
    ]

    ent_rows += col_stats(
        df_pre["entailment_count"],
        "Kalimat Entailment per Dok"
    )

    ent_rows += col_stats(
        df_pre["entailment_percentage"],
        "Rasio Entailment (%)"
    )

    ent_rows += col_stats(
        df_pre["avg_confidence"],
        "Confidence IndoBERT"
    )

    print_table(
        "STATISTIK TEXTUAL ENTAILMENT",
        ent_rows
    )

    ratio_rows = []

    for ratio in RATIOS:
        dfr = pd.DataFrame(results_by_ratio_first[ratio])

        ratio_rows.append([
            f"{ratio*100:.0f}%",
            len(dfr),
            f"{dfr['entailment_selected'].mean():.2f}",
            f"{dfr['post_sent_count'].mean():.2f}",
            f"{dfr['post_word_count'].mean():.2f}",
            f"{dfr['critical_fact_coverage'].mean():.4f}",
            f"{dfr['lexical_relevance'].mean():.4f}",
            f"{dfr['avg_entailment_support'].mean():.4f}",
            f"{dfr['quality_accepted'].mean()*100:.2f}%"
        ])

    print(
        "\n" + "=" * 130
    )

    print(tabulate(
        ratio_rows,
        headers=[
            "Rasio",
            "Dok",
            "Avg Kal. Dipilih",
            "Avg Kal. Ringkasan",
            "Avg Kata",
            "Fact Coverage",
            "Relevance",
            "Entailment Support",
            "Accepted"
        ],
        tablefmt="fancy_grid",
        colalign=("center",) * 9
    ))

In [ ]:
# ============================================================
# 18. PREVIEW HASIL SEA LLM
# ============================================================

if RUN_FULL_EXPERIMENT:
    for ratio in RATIOS:
        label = f"{ratio*100:.0f}%"
        dfr = pd.DataFrame(results_by_ratio_first[ratio])

        print("\n" + "=" * 100)
        print(f"PREVIEW SEA LLM — RASIO {label}")
        print("=" * 100)

        for _, row in dfr.head(3).iterrows():
            print(f"\nDOC_ID : {row['doc_id']}")
            print(f"TITLE  : {row['title']}")
            print(f"EVIDENCE: {row['evidence'][:500]}...")
            print(f"SUMMARY : {row['abstractive_summary']}")
            print(
                f"CHECK   : "
                f"coverage={row['critical_fact_coverage']:.3f} | "
                f"relevance={row['lexical_relevance']:.3f} | "
                f"support={row['avg_entailment_support']:.3f} | "
                f"accepted={row['quality_accepted']}"
            )
            print("-" * 100)

## 19. Diagnostic quality checks

Angka berikut **bukan pengganti ROUGE, BERTScore, atau human evaluation**. Fungsinya sebagai *quality-control diagnostic* untuk melihat apakah model cenderung:

- mempertahankan fakta penting,
- menghasilkan kalimat yang didukung evidence,
- tetap relevan terhadap evidence,
- dan lolos *guardrail*.

Untuk skripsi, *human evaluation* tetap digunakan untuk menilai **relevansi, faktualitas, kelengkapan, dan keterbacaan/koherensi**.

In [ ]:
# ============================================================
# 19. QUALITY-CONTROL AGGREGATION
# ============================================================

if RUN_FULL_EXPERIMENT:
    qc_rows = []

    for ratio in RATIOS:
        dfr = pd.DataFrame(results_by_ratio_first[ratio])

        qc_rows.append([
            f"{ratio*100:.0f}%",
            f"{dfr['critical_fact_coverage'].mean():.4f}",
            f"{dfr['lexical_relevance'].mean():.4f}",
            f"{dfr['avg_entailment_support'].mean():.4f}",
            int(dfr["weak_sentence_count"].sum()),
            f"{dfr['quality_accepted'].mean()*100:.2f}%"
        ])

    print(tabulate(
        qc_rows,
        headers=[
            "Rasio",
            "Critical Fact Coverage",
            "Lexical Relevance",
            "Entailment Support",
            "Weak Sentences",
            "Quality Accepted"
        ],
        tablefmt="fancy_grid",
        colalign=("center",) * 6
    ))

In [ ]:
# ============================================================
# 20. OPTIONAL: INSPEKSI MANUAL SATU DOKUMEN
# ============================================================

def inspect_document(doc_id: str, ratio: float = 1.0):
    if not RUN_FULL_EXPERIMENT:
        print("Jalankan full experiment terlebih dahulu.")
        return

    cache = next(
        (
            x for x in entailment_cache_first_liputan
            if x["doc_id"] == doc_id
        ),
        None
    )

    if cache is None:
        print(f"doc_id={doc_id} tidak ditemukan di cache.")
        return

    result = summarize_with_ratio(cache, ratio)

    print("\nTITLE")
    print(cache["title"])

    print("\nPREMISE")
    print(cache["premise"])

    print("\nSELECTED EVIDENCE")
    print(result["evidence"])

    print("\nSEA LLM SUMMARY")
    print(result["abstractive_summary"])

    print("\nQUALITY CHECK")
    print({
        "critical_fact_coverage": result["critical_fact_coverage"],
        "lexical_relevance": result["lexical_relevance"],
        "avg_entailment_support": result["avg_entailment_support"],
        "quality_accepted": result["quality_accepted"]
    })

# Contoh:
# inspect_document("liputan6_00002", 1.0)

## 21. Catatan metodologis untuk skripsi

Konfigurasi ini mempertahankan desain penelitian awal:

**Premise = first sentence → IndoBERT textual entailment → selected sentences → SeaLLM paraphrase/abstractive generation.**

Perubahan model hanya terjadi pada komponen generasi, sehingga perbandingan dengan eksperimen IndoT5 dapat diposisikan sebagai **perbandingan model abstraktif**, selama dataset, premise strategy, extraction ratio, dan prosedur evaluasi dibuat konsisten.

Guardrail tambahan:
- `critical_fact_coverage` = diagnostik preservasi fakta eksplisit.
- `lexical_relevance` = diagnostik keterikatan leksikal terhadap evidence.
- `avg_entailment_support` = diagnostik dukungan entailment terhadap kalimat hasil.
- `quality_accepted` = indikator internal untuk *quality control*, bukan metrik penelitian utama.

Untuk pelaporan ilmiah, hasil akhir tetap dievaluasi dengan **ROUGE, BERTScore, entailment/factuality measure, dan human evaluation** sesuai rancangan penelitian.